Nombre: David Díaz Paz y Puente | Matrícula: 650794

# A2.3 Modelos de ensamble, SVM y redes neuronales

---

## Introducción

En esta actividad se analiza el comportamiento de distintos modelos avanzados de clasificación sobre el mismo conjunto de datos utilizado en actividades previas. En particular, se entrenan y comparan modelos basados en ensambles, márgenes máximos y redes neuronales, con el propósito de diagnosticar el estrato socioconómico de una familia dadas sus condiciones de vivienda como _Bajo_ (1), _Medio bajo_ (2), _Medio alto_ (3) y _Alto_ (4). 

Los modelos considerados en este reporte son Random Forest, Boosting, Support Vector Machine (SVM) y una red neuronal sencilla. La intención no es realizar una optimización exhaustiva de hiperparámetros, sino construir modelos razonables, evaluar su desempeño con métricas de clasificación y reflexionar sobre sus diferencias en términos de precisión, estabilidad, interpretabilidad y complejidad.

El análisis se desarrolla de forma que el lector pueda comprender la metodología, los resultados y las conclusiones sin necesidad de revisar el código fuente. Por ello, se presentan explicaciones en texto, tablas y figuras comparativas que permiten interpretar el comportamiento de cada modelo de manera clara.

---

## Preparación del escenario experimental

### Carga de librerías

In [91]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

### Carga del dataset

In [61]:
df = pd.read_csv("features_lasso.csv")
df.head()

,cocina,cocina_dor,num_cuarto,bano_comp,bano_excus,bano_regad,estim_pago,tot_resid,tipo_viv_4,mat_pared_7,...,eli_basura_4,eli_basura_5,eli_basura_6,tenencia_2,tenencia_4,tenencia_5,escrituras_4,escrituras_5,prop_muj,est_socio
0,1,2.0,4,1,0,0,3000.0,2,False,False,...,False,False,False,False,True,False,False,False,0.50,2
1,1,2.0,4,1,0,0,2500.0,4,False,False,...,False,False,False,False,False,False,False,True,0.75,2
2,1,2.0,5,2,0,0,3000.0,4,False,False,...,False,False,False,False,True,False,False,False,0.75,2
3,1,2.0,3,1,0,0,3000.0,2,False,False,...,False,False,False,False,True,False,False,False,0.50,2
4,1,2.0,4,1,0,0,2500.0,6,False,True,...,False,False,False,False,False,False,False,True,0.50,2


### Descripción del dataset

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3767 entries, 0 to 3766
Data columns (total 45 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   cocina        3767 non-null   int64  
 1   cocina_dor    3767 non-null   float64
 2   num_cuarto    3767 non-null   int64  
 3   bano_comp     3767 non-null   int64  
 4   bano_excus    3767 non-null   int64  
 5   bano_regad    3767 non-null   int64  
 6   estim_pago    3767 non-null   float64
 7   tot_resid     3767 non-null   int64  
 8   tipo_viv_4    3767 non-null   bool   
 9   mat_pared_7   3767 non-null   bool   
 10  mat_pared_8   3767 non-null   bool   
 11  mat_techos_3  3767 non-null   bool   
 12  mat_techos_4  3767 non-null   bool   
 13  mat_techos_6  3767 non-null   bool   
 14  mat_techos_8  3767 non-null   bool   
 15  mat_pisos_3   3767 non-null   bool   
 16  lugar_coc_4   3767 non-null   bool   
 17  lugar_coc_5   3767 non-null   bool   
 18  lugar_coc_6   3767 non-null 

El conjunto de datos utilizado en esta actividad está conformado por un total de **3767 observaciones**, las cuales representan registros asociados a características de vivienda y condiciones socioeconómicas. Cada observación describe diferentes atributos relacionados con la infraestructura del hogar, disponibilidad de servicios básicos, materiales de construcción, equipamiento y condiciones de tenencia de la vivienda.

En cuanto a la estructura del dataset, se cuenta con **45 variables en total**, de las cuales **44 corresponden a variables predictoras** y **una variable corresponde a la variable objetivo**, denominada ``est_socio``. Esta variable representa la clasificación socioeconómica (_Bajo_ (1), _Medio bajo_ (2), _Medio alto_ (3) y _Alto_ (4)) asociada a cada registro y constituye el elemento principal que se busca predecir mediante los modelos de aprendizaje supervisado.

### División entrenamiento-prueba

In [63]:
# Cambia "target" por el nombre real de tu variable objetivo
X = df.drop(columns=["est_socio"])
y = df["est_socio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape, y_train.shape)
print("Prueba:", X_test.shape, y_test.shape)

Entrenamiento: (2636, 44) (2636,)
Prueba: (1131, 44) (1131,)


---

## Construcción de modelos

### Random Forest

In [64]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    random_state=42
)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'f1_macro', 'f1_weighted']
cv_rf = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=kfold,
    scoring=scoring
)

scores_rf = pd.DataFrame(cv_rf)
scores_rf = scores_rf.iloc[:, -3:]
print(scores_rf, "\n")
print(scores_rf.mean())

rf_model.fit(X_train, y_train)

   test_accuracy  test_f1_macro  test_f1_weighted
0       0.732955       0.638092          0.715465
1       0.730550       0.646124          0.706768
2       0.715370       0.634790          0.692690
3       0.707780       0.615364          0.691320
4       0.726755       0.646817          0.707320 

test_accuracy       0.722682
test_f1_macro       0.636237
test_f1_weighted    0.702713
dtype: float64


,n_estimators,500
,criterion,'gini'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### Boosting

In [ ]:
babc = AdaBoostClassifier(estimator=DTC(max_depth=15), n_estimators=250)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'f1_macro', 'f1_weighted']
cv_rf = cross_validate(
    babc,
    X_train,
    y_train,
    cv=kfold,
    scoring=scoring
)

scores_babc = pd.DataFrame(cv_rf)
scores_babc = scores_babc.iloc[:, -3:]
print(scores_babc, "\n")
print(scores_babc.mean())

babc.fit(X_train,y_train)

   test_accuracy  test_f1_macro  test_f1_weighted
0       0.693182       0.632324          0.692497
1       0.671727       0.588924          0.656415
2       0.673624       0.617579          0.670668
3       0.620493       0.551993          0.620273
4       0.658444       0.586063          0.647636 

test_accuracy       0.663494
test_f1_macro       0.595377
test_f1_weighted    0.657498
dtype: float64


,estimator,DecisionTreeC...(max_depth=15)
,n_estimators,250
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,None
,criterion,'gini'
,splitter,'best'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


### Support Vector Machine (SVM)

In [94]:
# svm_model = SVC(kernel="rbf", C=1)

df_svm = make_pipeline(StandardScaler(), svm.SVC(kernel='rbf', C=1.0, gamma='auto'))

scores_svm = cross_val_score(
    df_svm,
    X_train,
    y_train,
    cv=kfold,
    scoring="accuracy"
)

# scores_svm = pd.DataFrame(scores_svm)
# scores_svm = scores_svm.iloc[:, -3:]
print(scores_svm, "\n")
print(scores_svm.mean())

df_svm.fit(X_train, y_train)

[0.72916667 0.73624288 0.71726755 0.68690702 0.70208729] 

0.7143342820999368


,steps,"[('standardscaler', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'auto'



### Red Neuronal

---

## Evaluación y comparación de desempeño

### Random Forest

### Boosting

### Support Vector Machine (SVM)

### Red Neuronal

---

## Análsis crítico

--- 

## Conclusiones